# One ocean, unequal freshwater security

Pacific islands are surrounded by salt water, but homes, farms and public services depend on freshwater. **Island Signals** asks: **Where can the water safety chain break, from source to safe return?**

Six official challenge datasets are used. Surface temperature provides context. The main evidence comes from sea-surface temperature, sea level, rainfall, safely managed drinking-water access and the meteorological monitoring network. One open Pacific Community SDG 6 extract adds a water-quality measure: the proportion of domestic wastewater safely treated in nine territory reports from 2024.

These comparisons do not form a causal model. They show a shared physical pressure alongside different freshwater conditions, service levels and formal observation.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path('data/source/challenge-2026')
ADDITIONAL_DATA_DIR = Path('data/source/additional')
OUTPUT_DIR = Path('analysis/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NAME_MAP = {
    'Micronesia (Federated States of)': 'Federated States of Micronesia',
    'Micronesia, Federated State of': 'Federated States of Micronesia',
}

def load_series(filename):
    frame = pd.read_csv(DATA_DIR / filename).rename(columns={
        'Pacific Island Countries and territories': 'territory',
        'TIME_PERIOD': 'year',
        'OBS_VALUE': 'value',
    })
    frame['territory'] = frame['territory'].replace(NAME_MAP)
    frame['year'] = pd.to_numeric(frame['year'], errors='coerce')
    frame['value'] = pd.to_numeric(frame['value'], errors='coerce')
    frame = frame[['territory', 'year', 'value']].dropna().sort_values(['territory', 'year'])
    duplicates = frame.duplicated(['territory', 'year']).sum()
    if duplicates:
        raise ValueError(f'{filename}: {duplicates} duplicate territory-year rows')
    return frame

def fitted_slope(group, multiplier=1):
    return np.polyfit(group['year'], group['value'], 1)[0] * multiplier

def trend_table(frame, multiplier=1):
    return (frame.groupby('territory')
        .apply(lambda group: pd.Series({
            'first_year': int(group.year.min()),
            'last_year': int(group.year.max()),
            'observations': len(group),
            'slope': fitted_slope(group, multiplier),
        }), include_groups=False)
        .reset_index())

## 1. The ocean is the common pressure

Sea-surface temperature and sea level answer different questions and use different units. Keeping them separate makes their shared direction more meaningful.

In [2]:
surface = load_series('mean-surface-temperature-anomalies.csv')
sst = load_series('mean-sea-surface-temperature-anomalies.csv')
sea_level = load_series('sea-level-anomalies.csv')

surface_trends = trend_table(surface, 100).assign(indicator='Surface temperature', unit='°C per century')
sst_trends = trend_table(sst, 100).assign(indicator='Sea-surface temperature', unit='°C per century')
sea_level_trends = trend_table(sea_level, 1000).assign(indicator='Sea level', unit='millimetres per year')

ocean_summary = pd.concat([surface_trends, sst_trends, sea_level_trends], ignore_index=True)
(ocean_summary.groupby(['indicator', 'unit'])
 .agg(territories=('territory', 'nunique'), positive=('slope', lambda values: int((values > 0).sum())),
      minimum=('slope', 'min'), median=('slope', 'median'), maximum=('slope', 'max'))
 .round(3))

                                              territories  ...  maximum
indicator               unit                               ...         
Sea level               millimetres per year           21  ...    5.403
Sea-surface temperature °C per century                 21  ...    0.414
Surface temperature     °C per century                 22  ...    0.438

[3 rows x 5 columns]

All 22 surface-temperature series have positive fitted trends. The 21 sea-surface-temperature series and 21 sea-level series also point upward. The direction is shared, but the result does not mean every year rose or every territory faces the same exposure. These records do not measure saltwater intrusion or aquifer condition.

## 2. Rain is where the regional story breaks

Many island freshwater systems depend on rainfall. Two parts of the record matter here: the fitted direction from 1979 to 2025 and the size of the annual swings around the reference average.

In [3]:
rainfall = load_series('rainfall-anomalies.csv')
rainfall_summary = (rainfall.groupby('territory')
    .apply(lambda group: pd.Series({
        'first_year': int(group.year.min()),
        'last_year': int(group.year.max()),
        'trend_per_year': fitted_slope(group),
        'annual_variability': group.value.std(ddof=1),
        'largest_absolute_anomaly': group.value.abs().max(),
    }), include_groups=False)
    .reset_index())

print('Upward fitted trends:', int((rainfall_summary.trend_per_year > 0).sum()))
print('Downward fitted trends:', int((rainfall_summary.trend_per_year < 0).sum()))
rainfall_summary.sort_values('annual_variability', ascending=False).head(8).round(2)

Upward fitted trends: 15
Downward fitted trends: 7


                         territory  ...  largest_absolute_anomaly
8                            Nauru  ...                      66.6
6                         Kiribati  ...                      50.3
17                         Tokelau  ...                      38.8
19                          Tuvalu  ...                      41.5
2   Federated States of Micronesia  ...                      44.4
12                           Palau  ...                      31.6
9                    New Caledonia  ...                      37.3
20                         Vanuatu  ...                      33.7

[8 rows x 6 columns]

Fifteen fitted rainfall trends point upward and seven point downward. Annual variability also differs substantially. A Pacific average would hide the freshwater conditions that storage, drought and drainage decisions depend on. Neither direction is automatically good or bad.

**Related study.** White, Falkland and Redfern studied observations from Tarawa and Kiritimati, Kiribati, from 1951 to 2023. They found significant ocean warming but no significant long-term trend in annual rainfall. ENSO variability remained strong, and severe drought remained a freshwater challenge. Their study supports reading rainfall locally, but it is not part of the challenge-dataset calculation above. [White, Falkland and Redfern (2024)](https://doi.org/10.3390/atmos15060666).

## 3. Freshwater security begins from unequal access

The 2020 comparison keeps every territory in the same year. Safely managed drinking water means an improved source that is accessible on the premises, available when needed and free from contamination.

In [4]:
safe_water = load_series('proportion-of-population-using-safely-managed-drinking-water-services.csv')
water_2020 = safe_water.loc[safe_water.year.eq(2020), ['territory', 'value']].sort_values('value')

print('Territories represented:', len(water_2020))
print('Range:', f'{water_2020.value.min():.2f}% to {water_2020.value.max():.2f}%')
print('Below 70%:', int((water_2020.value < 70).sum()))
water_2020

Territories represented: 19
Range: 48.11% to 100.00%
Below 70%: 3


                          territory   value
132                Papua New Guinea   48.11
110                 Solomon Islands   67.30
381               Wallis and Futuna   68.89
290                        Kiribati   74.31
404                French Polynesia   83.52
154                Marshall Islands   85.49
359                  American Samoa   89.85
89   Federated States of Micronesia   90.11
221                         Vanuatu   90.49
336        Northern Mariana Islands   90.64
20                             Fiji   95.42
427                   New Caledonia   96.64
43                             Niue   97.01
244                           Samoa   97.40
177                           Tonga   98.74
313                          Tuvalu   99.22
66                            Palau   99.56
267                    Cook Islands   99.97
200                           Nauru  100.00

Access ranges from 48.11% to 100% across 19 territories, with three below 70%. That is the difference between water surrounding an island and water a household can use safely. It is also an existing service condition, not an outcome that can be attributed to the climate trends in this notebook.

In [5]:
wastewater = pd.read_csv(ADDITIONAL_DATA_DIR / 'spc-sdg6-wastewater-treatment-2024.csv')
wastewater['year'] = pd.to_numeric(wastewater['year'], errors='raise')
wastewater['proportion_safely_treated'] = pd.to_numeric(wastewater['proportion_safely_treated'], errors='raise')

print('2024 territory reports:', len(wastewater))
print('Range:', f'{wastewater.proportion_safely_treated.min():.1f}% to {wastewater.proportion_safely_treated.max():.1f}%')
wastewater.sort_values('proportion_safely_treated')

2024 territory reports: 9
Range: 7.3% to 79.1%


          territory  year  proportion_safely_treated     unit
4  Papua New Guinea  2024                       7.29  percent
8           Vanuatu  2024                      13.21  percent
6             Tonga  2024                      30.16  percent
2          Kiribati  2024                      32.90  percent
3             Nauru  2024                      35.51  percent
1              Fiji  2024                      39.88  percent
7            Tuvalu  2024                      42.73  percent
5             Samoa  2024                      43.81  percent
0    American Samoa  2024                      79.06  percent

A second water measure changes the practical reading. The nine territory reports from 2024 range from 7.29% to 79.06% for domestic wastewater safely treated. This extract is not a complete Pacific ranking, but it shows that water security has two gates: safe water entering homes and safe management after use. The drinking-water comparison is from 2020, while this treatment series is from 2024. They should be read as separate checks, without treating one as the cause of the other.

## 4. Local decisions need local observations

The official indicator counts fixed land climate-observation stations that comply with World Meteorological Organization standards. It does not include every instrument or source of weather information available to a territory.

In [6]:
stations = load_series('meteorological-monitoring-network.csv')
stations_2026 = stations.loc[stations.year.eq(2026), ['territory', 'value']].sort_values('value')

print('Territories represented:', len(stations_2026))
print('Range:', f'{stations_2026.value.min():.0f} to {stations_2026.value.max():.0f} stations')
print('Reporting zero:', int((stations_2026.value == 0).sum()))
print('Reporting one or fewer:', int((stations_2026.value <= 1).sum()))
stations_2026

Territories represented: 18
Range: 0 to 8 stations
Reporting zero: 3
Reporting one or fewer: 5


                           territory  value
916                             Niue      0
389                         Pitcairn      0
1306                           Nauru      0
1572                         Tokelau      1
778                            Palau      1
699                            Samoa      2
1385                    Cook Islands      2
1561                Marshall Islands      2
1011                          Tuvalu      3
1649                 Solomon Islands      3
561   Federated States of Micronesia      3
475                    New Caledonia      4
1168                        Kiribati      4
80                             Tonga      4
1087                Papua New Guinea      6
251                          Vanuatu      6
177                 French Polynesia      7
1479                            Fiji      8

The 2026 comparison covers 18 territories and ranges from zero to eight compliant fixed land stations. Three report zero and five report one or fewer. Territory size, island dispersion and observing needs differ, so the count cannot judge adequacy. It does show that the formal observing base is uneven where local freshwater evidence matters most.

## 5. What the records say

The result is not simply that rainfall differs. The most local part of the freshwater problem is also the least uniform.

1. **Pressure:** every observed sea-surface-temperature and sea-level fitted trend points upward.
2. **Supply:** rainfall trends split and annual variability differs.
3. **Access:** safely managed drinking-water access spans 48.11% to 100% in the common 2020 comparison.
4. **After use:** domestic wastewater safely treated ranges from 7.29% to 79.06% across nine 2024 territory reports.
5. **Observation:** WMO-compliant fixed land station counts range from zero to eight among reporting territories.

Taken together, the records point to a water safety watch: protect coastal and freshwater sources, use territory-level rainfall and storage evidence, keep drinking-water services safe, treat wastewater before discharge, and maintain the observations needed to update the picture. The two water measures cover different years and different parts of the system, so they are checkpoints for planning rather than a combined score. This analysis identifies a practical response, but it does not test how effective any intervention would be.

Research from Fiji, Vanuatu and Solomon Islands reaches a similar conclusion: sustained rural water safety planning has to fit local governance, community management and ways of sharing knowledge. [Souter et al. (2024)](https://doi.org/10.2166/wh.2024.144).

**A shared ocean warning calls for a complete local water-safety plan.**

The useful result is a set of check points for action: source, supply, service, treatment and observation. The measures are kept separate because they describe different parts of that chain.

### References

- White, I., Falkland, T. and Redfern, F. (2024). *Ocean Surface Warming and Long-Term Variability in Rainfall in Equatorial Pacific Atolls*. Atmosphere, 15(6), 666. https://doi.org/10.3390/atmos15060666
- Souter, R. T. C. et al. (2024). *Strengthening rural community water safety planning in Pacific Island countries: evidence and lessons from Solomon Islands, Vanuatu, and Fiji*. Journal of Water and Health, 22(3), 467–486. https://doi.org/10.2166/wh.2024.144
- Pacific Community. *Sustainable Development Goal 06: Clean Water and Sanitation data*, SPC:DF_SDG_06(4.4), modified 6 August 2026. The notebook uses the nine-row 2024 extract in `data/source/additional/spc-sdg6-wastewater-treatment-2024.csv`, normalized from the SPC reference area label `Naoero` to `Nauru`. https://pacificdata.org/data/dataset/sustainable-development-goal-06-clean-water-and-sanitation-df-sdg-06

In [7]:
story_summary = (rainfall_summary
    .merge(water_2020.rename(columns={'value': 'safe_water_2020'}), on='territory', how='outer')
    .merge(wastewater[['territory', 'proportion_safely_treated']], on='territory', how='outer')
    .merge(stations_2026.rename(columns={'value': 'wmo_fixed_land_stations_2026'}), on='territory', how='outer'))
story_summary.to_csv(OUTPUT_DIR / 'water_story_summary.csv', index=False)
story_summary.sort_values('territory')

                         territory  ...  wmo_fixed_land_stations_2026
0                   American Samoa  ...                           NaN
1                     Cook Islands  ...                           2.0
2   Federated States of Micronesia  ...                           3.0
3                             Fiji  ...                           8.0
4                 French Polynesia  ...                           7.0
5                             Guam  ...                           NaN
6                         Kiribati  ...                           4.0
7                 Marshall Islands  ...                           2.0
8                            Nauru  ...                           0.0
9                    New Caledonia  ...                           4.0
10                            Niue  ...                           0.0
11        Northern Mariana Islands  ...                           NaN
12                           Palau  ...                           1.0
13                Pa